# COMP4040 — Spark Lab: Word Count (RDD)
This lab is classic Spark MOOC WordCount lab. You will build a full Word Count pipeline:
**`textFile → map/flatMap → (word, 1) → reduceByKey → takeOrdered`**. We’ll run Word Count on a **real public dataset** (Shakespeare from Project Gutenberg).


## 0) Setup
Works on **Colab** or **local Jupyter**. **If local:** You need Java installed for Spark to run.


In [1]:
import sys, importlib
def _has(pkg):
    try:
        importlib.import_module(pkg)
        return True
    except Exception:
        return False

if not _has('pyspark'):
    print('Installing pyspark...')
    !{sys.executable} -m pip -q install pyspark
else:
    print('pyspark already available ✅')

Installing pyspark...


'c:\Users\jimmy\Documents\Ta�i' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName('COMP4040-WordCount-Lab-Assert')
         .master('local[*]')
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel('ERROR')

print('Spark version:', spark.version)
print('Default parallelism:', sc.defaultParallelism)

In [ ]:
# CHECK 0
assert sc.parallelize([1,2,3,4]).sum() == 10
print('✅ Spark is working')

## Cheat-sheet
RDD basics:
- Create: `sc.parallelize(list)` or `sc.textFile(path)`
- Transformations (lazy):
  - `rdd.map(f)`
  - `rdd.flatMap(f)`  (1 input → many outputs)
  - `rdd.filter(pred)`
  - `rdd.reduceByKey(f)` (pair RDD only)
- Actions (trigger jobs): `count()`, `collect()`, `take(n)`, `takeOrdered(n, key=...)`

Official references (recommended reading after lab):
- Spark RDD Programming Guide: https://spark.apache.org/docs/latest/rdd-programming-guide.html
- `reduceByKey` API (PySpark): https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.reduceByKey.html

> Note: `reduceByKey` requires the reduce function to be **associative and commutative**.


## Part 1 — Warm-up: Base RDD, `map`, Pair RDD
We start with a tiny toy dataset to practice Spark syntax before scaling up.


In [ ]:
wordsList = ['cat', 'elephant', 'rat', 'rat', 'cat']
wordsRDD = sc.parallelize(wordsList, 4)
print('numPartitions =', wordsRDD.getNumPartitions())
print('wordsRDD.take(5) =', wordsRDD.take(5))

### (1a) Function + map
Implement `makePlural(word)` that returns the plural form by adding `'s'`.
Then apply it to the RDD using `map`.


In [ ]:
# TODO: implement makePlural and create pluralRDD
def makePlural(word):
    """Adds an 's' to `word`."""
    # TODO
    return None

pluralRDD = None  # TODO: wordsRDD.map(...)
print(pluralRDD.collect() if pluralRDD is not None else pluralRDD)

In [ ]:
# CHECK 1a
assert makePlural('rat') == 'rats'
assert pluralRDD.collect() == ['cats', 'elephants', 'rats', 'rats', 'cats']
print('✅ Check 1a passed')

### (1b) Pair RDD
Create a pair RDD of `(word, 1)` for each word.

**Hint:** `wordsRDD.map(lambda w: (w, 1))`


In [ ]:
# TODO: create wordPairs
wordPairs = None
print(wordPairs.collect() if wordPairs is not None else wordPairs)

In [ ]:
# CHECK 1b
assert wordPairs.collect() == [('cat', 1), ('elephant', 1), ('rat', 1), ('rat', 1), ('cat', 1)]
print('✅ Check 1b passed')

## Part 2 — Counting words (pair RDD)
We’ll implement counting with `reduceByKey`.

**Idea:** map each word → `(word, 1)` then sum by key.


In [ ]:
# TODO: compute wordCounts using reduceByKey
wordCounts = None
print(wordCounts.collect() if wordCounts is not None else wordCounts)

In [ ]:
# CHECK 2
assert sorted(wordCounts.collect()) == [('cat', 2), ('elephant', 1), ('rat', 2)]
print('✅ Check 2 passed')

### (2b) Unique words + average count
Compute:
- `uniqueWords`: number of unique words
- `average`: mean count per unique word

**Hints:**
- Unique words: `wordCounts.count()`
- Sum all counts: `wordCounts.map(lambda kv: kv[1]).sum()` or `reduce`
- Mean: `total / unique`


In [ ]:
# TODO: compute uniqueWords, totalCount, average
uniqueWords = None
totalCount = None
average = None
print(uniqueWords, totalCount, average)

In [ ]:
# CHECK 2b
assert uniqueWords == 3
assert round(average, 2) == 1.67
print('✅ Check 2b passed')

## Part 3 — Real dataset download (Project Gutenberg)
Dataset: **The Complete Works of William Shakespeare** (Project Gutenberg)
- Official book page: https://www.gutenberg.org/ebooks/100

The cell below:
1) Creates a `data/` folder
2) Downloads the text file if it does not exist
3) Prints file size

> If download fails (some networks block Gutenberg), try re-run the cell. If it still fails,
  you can replace the URL with the GitHub mirror shown in the code comments.


In [ ]:
import os
from pathlib import Path
import urllib.request

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

# Primary source (Gutenberg cache plain text)
URL = 'https://www.gutenberg.org/cache/epub/100/pg100.txt'

FILE_PATH = DATA_DIR / 'shakespeare_pg100.txt'

if not FILE_PATH.exists():
    print('Downloading dataset...')
    req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as r, open(FILE_PATH, 'wb') as f:
        f.write(r.read())
    print('Downloaded to', FILE_PATH)
else:
    print('Found existing file:', FILE_PATH)

print('File size (MB) =', round(FILE_PATH.stat().st_size / 1024 / 1024, 2))

In [ ]:
# CHECK 3 (dataset exists)
assert FILE_PATH.exists()
assert FILE_PATH.stat().st_size > 1_000_000  # > 1MB
print('✅ Dataset ready')

## Part 4 — Word Count “app” on Shakespeare
Now we build reusable functions and run them at scale.

### (4a) Define `wordCount(wordListRDD)`
Input: RDD of words (strings)
Output: Pair RDD of `(word, count)`

**Hint:** `wordListRDD.map(lambda w: (w, 1)).reduceByKey(...)`


In [ ]:
# TODO: implement wordCount
def wordCount(wordListRDD):
    """Return (word, count) pairs from an RDD of words."""
    # TODO
    return None

print(wordCount(wordsRDD).collect() if wordCount(wordsRDD) is not None else None)

In [ ]:
# CHECK 4a
assert sorted(wordCount(wordsRDD).collect()) == [('cat', 2), ('elephant', 1), ('rat', 2)]
print('✅ Check 4a passed')

### (4b) Clean a line of text
Define `removePunctuation(text)` that:
- lowercases
- removes punctuation / non-alphanumeric (keep spaces)
- strips leading/trailing spaces

**Example:**
`" The Elephant's 4 cats. "` → `"the elephants 4 cats"`

**Hint:** `re.sub(r"[^a-z0-9\s]", "", text.lower())`


In [ ]:
# TODO: implement removePunctuation
import re

def removePunctuation(text: str) -> str:
    # TODO
    return None

print(removePunctuation(" The Elephant's 4 cats. "))

In [ ]:
# CHECK 4b
assert removePunctuation(" The Elephant's 4 cats. ") == 'the elephants 4 cats'
print('✅ Check 4b passed')

### (4c) Load dataset with `sc.textFile` and clean lines
We load the file as an RDD of lines, then apply `removePunctuation`.

**Syntax reminder:**
- `rdd = sc.textFile(path, minPartitions=8)`
- `rdd2 = rdd.map(f)`


In [ ]:
shakespeareRDD = (sc
                 .textFile(str(FILE_PATH), minPartitions=8)
                 .map(removePunctuation))

print('numPartitions =', shakespeareRDD.getNumPartitions())
print('numLines =', shakespeareRDD.count())
print('Sample lines:')
for line in shakespeareRDD.take(5):
    print('  ', line[:120])

In [ ]:
# CHECK 4c
numLines = shakespeareRDD.count()
assert numLines > 10000
print('✅ Check 4c passed')

### (4d) Convert lines → words (flatMap)
We split each line into words.

**Important for this lab:** Use `split(' ')` (single-space split) to show why empty tokens appear.
Later you will filter `''`.
Hint: `rdd.flatMap(lambda line: line.split(' '))`


In [ ]:
# TODO: create shakespeareWordsRDD and shakespeareWordCount
shakespeareWordsRDD = None
shakespeareWordCount = None
print(shakespeareWordCount)

In [ ]:
# CHECK 4d (flow check, not exact)
assert shakespeareWordsRDD is not None
assert shakespeareWordCount is not None
assert shakespeareWordCount > 900_000
print('✅ Check 4d passed')

### (4e) Remove empty tokens
Filter out empty strings `''`.

**Hint:** `rdd.filter(lambda w: w != '')`


In [ ]:
# TODO: create shakeWordsRDD and shakeWordCount
shakeWordsRDD = None
shakeWordCount = None
print(shakeWordCount)

In [ ]:
# CHECK 4e
assert shakeWordsRDD is not None
assert shakeWordCount is not None
assert shakeWordCount < shakespeareWordCount
assert shakeWordCount > 900_000
print('✅ Check 4e passed')

### (4f) Count words + top 15
Compute counts and show the top 15 most frequent words.

**Hints:**
- `counts = wordCount(shakeWordsRDD)`
- `top15 = counts.takeOrdered(15, key=lambda kv: -kv[1])`


In [ ]:
# TODO: compute top15WordsAndCounts
top15WordsAndCounts = None
print(top15WordsAndCounts)

In [ ]:
# CHECK 4f (flow check, not exact)
assert isinstance(top15WordsAndCounts, list) and len(top15WordsAndCounts) == 15
assert top15WordsAndCounts[0][0] == 'the'
assert top15WordsAndCounts[0][1] > 20000
print('✅ Check 4f passed')

## Optional: what to explore next (self-study)
1) Compare `groupByKey` vs `reduceByKey` on Shakespeare tokens (timing).
2) Remove stopwords and see top words change.
3) Change the tokenizer: should numbers like `1599` count as a token?

### Cleanup
Stop Spark when finished.


In [ ]:
spark.stop()